## Imports

In [19]:
import re
import os
import pickle
import ast
from collections import OrderedDict
from tqdm.auto import tqdm
from dotenv import load_dotenv
from minsearch import Index, VectorSearch
import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer

load_dotenv()

True

## Load chunked document

In [15]:
def export_object(obj, file_path):
    if not file_path.endswith('.pkl'):
        raise ValueError(f"File {file_path} is not a pickle file")
    
    with open(file_path, 'wb') as f:
        pickle.dump(obj, f)

In [16]:
def load_object(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File {file_path} does not exist")
    
    if not file_path.endswith('.pkl'):
        raise ValueError(f"File {file_path} is not a pickle file")
    
    with open(file_path, 'rb') as f:
        return pickle.load(f)

In [ ]:
simple_chunks = load_object('data/simple_chunks.pkl')
paragraph_chunks = load_object('data/paragraph_chunks.pkl')
section_chunks = load_object('data/section_chunks.pkl')
# intelligent_chunks = load_object('data/intelligent_chunks.pkl')

In [5]:
print(simple_chunks[0].keys())
print(simple_chunks[0])

dict_keys(['filename', 'chunk_content'])
{'filename': 'freecodecamp-main/.github/pull_request_template.md', 'chunk_content': 'Checklist:\n\n<!-- Please follow this checklist and put an x in each of the boxes, like this: [x]. It will ensure that our team takes your pull request seriously. -->\n\n- [ ] I have read and followed the [contribution guidelines](https://contribute.freecodecamp.org).\n- [ ] I have read and followed the [how to open a pull request guide](https://contribute.freecodecamp.org/how-to-open-a-pull-request/).\n- [ ] My pull request targets the `main` branch of freeCodeCamp.\n- [ ] I have tested these changes either locally on my machine, or GitHub Codespaces.\n\n<!--If your pull request closes a GitHub issue, replace the XXXXX below with the issue number.-->\n\nCloses #XXXXX\n\n<!-- Feel free to add any additional description of changes below this line -->'}


## Indexing chunks

In [6]:
def process_chunks(chunks):
    chunks_processed = [
        ast.literal_eval(chunk) for chunk in chunks
    ]
    return chunks_processed

In [7]:
def index_chunks(chunks,
                 text_fields,
                 keyword_fields=[]):
    if not isinstance(chunks, list):
        chunks = [chunks]
        
    if not all(isinstance(chunk, dict) for chunk in chunks):
        raise ValueError("All chunks must be dictionaries")
    
    index = Index(
        text_fields = text_fields,
        keyword_fields = keyword_fields
    )
    index.fit(chunks)
    return index

In [8]:
simple_index = index_chunks(simple_chunks, text_fields=["chunk_content", "filename"])
paragraph_index = index_chunks(paragraph_chunks, text_fields=["chunk_content", "filename"])
section_index = index_chunks(section_chunks, text_fields=["chunk_content", "filename"])
# intelligent_index = index_chunks(intelligent_chunks, text_fields=["chunk_content", "filename"])

## Lexical (keyword) Search

In [17]:
def lexical_search(index, query, n=5):
    return index.search(query, num_results=n)

In [10]:
lexical_search(simple_index, 
             query="Which Python libraries are good for Data Visualization?"
)

[{'title': 'Data Visualization',
  'superBlock': 'data-visualization',
  'certification': 'data-visualization',
  'filename': 'freecodecamp-main/client/src/pages/learn/data-visualization/index.md',
  'chunk_content': '## Introduction to Data Visualization\n\nThis is a stub introduction for Data Visualization'},
 {'title': 'Introduction to the Data Visualization Projects',
  'block': 'data-visualization-projects',
  'superBlock': 'data-visualization',
  'filename': 'freecodecamp-main/client/src/pages/learn/data-visualization/data-visualization-projects/index.md',
  'chunk_content': '## Introduction to the Data Visualization Projects\n\nThese challenges let you test your data visualization skills and how to transfer and use data using AJAX technologies.\n\nBy the end of this, you would have 5 projects to showcase your data visualization skills that you can show off to friends, family, employers, etc. Have fun and remember to use the [Read-Search-Ask](https://forum.freecodecamp.org/t/how-

In [11]:
lexical_search(section_index, 
             query="Which Python libraries are good for Data Visualization?"
)

[{'title': 'Data Visualization',
  'superBlock': 'data-visualization',
  'certification': 'data-visualization',
  'filename': 'freecodecamp-main/client/src/pages/learn/data-visualization/index.md',
  'chunk_content': '## Introduction to Data Visualization\n\nThis is a stub introduction for Data Visualization'},
 {'title': 'Introduction to the Data Visualization Projects',
  'block': 'data-visualization-projects',
  'superBlock': 'data-visualization',
  'filename': 'freecodecamp-main/client/src/pages/learn/data-visualization/data-visualization-projects/index.md',
  'chunk_content': '## Introduction to the Data Visualization Projects\n\nThese challenges let you test your data visualization skills and how to transfer and use data using AJAX technologies.\n\nBy the end of this, you would have 5 projects to showcase your data visualization skills that you can show off to friends, family, employers, etc. Have fun and remember to use the [Read-Search-Ask](https://forum.freecodecamp.org/t/how-

## Vector Search

In [12]:
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

In [13]:
query = "Which Python libraries are good for Data Visualization?"
query_embedding = embedding_model.encode(query)

In [14]:
chunk_embeddings = []
chunk_texts = []

for chunk in tqdm(section_chunks):
    chunk_text = f"""
    Filename: {chunk['filename']}\n
    Content:\n{chunk['chunk_content']}
    """
    chunk_embedding =np.array(embedding_model.encode(chunk_text))
    chunk_embeddings.append(chunk_embedding)
    chunk_texts.append(chunk_text)

  0%|          | 0/47063 [00:00<?, ?it/s]

In [18]:
def vector_search(index, query_emb, n=5):
    return index.search(query_emb, num_results=n)

In [22]:
vec_index = VectorSearch(keyword_fields=[])
vec_index.fit(chunk_embeddings, chunk_texts)
vector_search(vec_index, query_embedding)

['\n    Filename: freecodecamp-main/curriculum/challenges/english/blocks/lecture-introduction-to-python/67fe81c9c6fd3714343a45ad.md\n\n    Content:\n## --text--\n\nWhich libraries are commonly used for data analysis in Python?\n    ',
 '\n    Filename: freecodecamp-main/curriculum/challenges/english/blocks/learn-functions-and-graphing/63e1798f811fda1bc546bba0.md\n\n    Content:\n## --text--\n\nWhat Python library would you import to create arrays that you can graph?\n    ',
 '\n    Filename: ml-for-beginners-main/translations/en/2-regression/2-data/readme.md\n\n    Content:\n## Visualization Strategies\n\nA data scientist\'s role often involves demonstrating the quality and characteristics of the data they\'re working with. This is done by creating visualizations—plots, graphs, and charts—that reveal relationships and gaps that might otherwise be hard to identify.\n\n[![ML for beginners - How to Visualize Data with Matplotlib](https://img.youtube.com/vi/SbUkxH6IJo0/0.jpg)](https://yout

In [23]:
export_object(chunk_embeddings, "data/section_chunks_embeddings.pkl")
export_object(chunk_texts, "data/section_chunks_texts.pkl")
export_object(vec_index, "data/section_chunks_vec_index.pkl")
export_object(section_index, "data/section_chunks_lexical_index.pkl")

## Hybrid Search (Lexical + Semantic)

In [30]:

search_query = "Which Python libraries are good for Data Visualization?"
search_query_embedding = embedding_model.encode(search_query)

lexical_search_results = [result['chunk_content'] for result in lexical_search(section_index, search_query)]
vector_search_results = vector_search(vec_index, search_query_embedding)

final_search_results = list(OrderedDict.fromkeys(lexical_search_results + vector_search_results))

for i, result in enumerate(final_search_results):
    print(f"Result #{i + 1}:\n{result}\n")
    

Result #1:
## Introduction to Data Visualization

This is a stub introduction for Data Visualization

Result #2:
## Introduction to the Data Visualization Projects

These challenges let you test your data visualization skills and how to transfer and use data using AJAX technologies.

By the end of this, you would have 5 projects to showcase your data visualization skills that you can show off to friends, family, employers, etc. Have fun and remember to use the [Read-Search-Ask](https://forum.freecodecamp.org/t/how-to-get-help-when-you-are-stuck-coding/19514) method if you get stuck.

Result #3:
## --text--

Which libraries are commonly used for data analysis in Python?

Result #4:
## Introduction to the Data Visualization with D3 Challenges

D3.js, or D3, stands for Data Driven Documents. D3 is a JavaScript library to create dynamic and interactive data visualizations in the browser. It's built to work with common web standards, namely HTML, CSS, and Scalable Vector Graphics (SVG).<br>